In [1]:
from base import generator_haar

def init_experiment(n):
    d = 2**n

    # Generate 6^n density matrices
    rho_list = generator_haar.generate_n_qubits_rho_haar(n)
    print(f"Generated {len(rho_list)} of {rho_list[0].shape} rho.")

    # Generate unitary
    unitary = generator_haar.random_unitary(d)
    print(f"Generated {unitary.shape} unitary operators.")
    return rho_list, unitary

In [2]:
import numpy as np
import tensorflow as tf

from base import epsilon_rho
def calculate_rho2_unitary(rho_list, unitary):
    rho2_unitary = []
    for rho in rho_list:
        rho2_unitary.append(epsilon_rho.calculate_from_unitary(rho, unitary))
    return rho2_unitary

def calculate_rho2_dephasing(rho_list, n, gamma):
    rho2 = []
    for rho in rho_list:
        rho2.append(epsilon_rho.calculate_dephasing(rho, n, gamma))
    return rho2

def write_to_file(filename, data):
    """Write TensorFlow tensor data to a text file without truncation."""
    tensor_data = data.numpy() if isinstance(data, tf.Tensor) else data

    # Open the file and write the tensor data
    with open(filename, 'w') as f:
        if isinstance(data, np.ndarray):
            np.savetxt(f, data, fmt="%.6f")
        elif isinstance(data, list):
            for item in data:
                f.write(f"{item}\n")
        else:
            f.write(str(data))





In [3]:
import os
from base import optimize_algorithm
from base import metrics
experiment_folder = 'results/experiment_new/dephasing'

for num_qubits in range(2, 3):
    if (experiment_folder == ''):
        break
    else:
        write_folder = os.path.join(experiment_folder, str(num_qubits) + "_qubits")
        if not os.path.exists(write_folder):
            os.makedirs(write_folder)
    print(f"N={num_qubits}")

    #-----Init experiment-----
    rho_list, unitary = init_experiment(num_qubits)
    write_to_file(os.path.join(write_folder, "rho_list.txt"), rho_list)

    g_s = np.linspace(1, 10e-3, 20)
    for g in g_s:
        folder_path = os.path.join(write_folder, "_{:.2f}".format(g))
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

        rho2_list = calculate_rho2_dephasing(rho_list, num_qubits, g)
    
        #-----Learn kraus operators-----
        unitary_res, cost_dict = optimize_algorithm.optimize_adam_unitary_dagger_set(rho_list, rho2_list, unitary, 0.008, num_loop=200)
    
        #-----Calculate result data-----
        rho3_list = epsilon_rho.calculate_set_from_unitary_dagger(unitary_res, rho2_list)
        rho2_unitary_list = calculate_rho2_unitary(rho_list, unitary_res)
    
        mean_fidelity_rho_rho3 = metrics.mean_fidelity(rho3_list, rho_list)
        mean_fidelity_rho2_rho2 = metrics.mean_fidelity(rho2_unitary_list, rho2_list)

        #-----Write to folder-----    
        write_to_file(os.path.join(folder_path,"unitary.txt"), unitary)
        write_to_file(os.path.join(folder_path,"unitary_res.txt"), unitary_res)
        write_to_file(os.path.join(folder_path,"cost_dict.txt"), cost_dict)

        write_to_file(os.path.join(folder_path,"rho2_list.txt"), rho2_list)
        write_to_file(os.path.join(folder_path,"rho2_unitary_list.txt"), rho2_unitary_list)

        write_to_file(os.path.join(folder_path,"mean_fidelity_rho_rho3.txt"), mean_fidelity_rho_rho3.numpy())
        write_to_file(os.path.join(folder_path,"mean_fidelity_rho2_rho2.txt"), mean_fidelity_rho2_rho2.numpy())

        print(g, num_qubits)
        print(cost_dict[-1])

    
    

N=2
Generated 36 of (4, 4) rho.
Generated (4, 4) unitary operators.
1.0 2
0.5952069144588424
0.9478947368421052 2
0.4240244889878098
0.8957894736842105 2
0.3570860501139302
0.8436842105263158 2
0.301664158705534
0.791578947368421 2
0.24883778727091352
0.7394736842105263 2
0.22055695862540903
0.6873684210526316 2
0.1869578417412329
0.6352631578947369 2
0.1561718585234314
0.5831578947368421 2
0.13036546554842515
0.5310526315789473 2
0.10864537380013689
0.47894736842105257 2
0.09406974789473872
0.4268421052631579 2
0.07726477730483541
0.37473684210526315 2
0.06413157527341956
0.3226315789473684 2
0.047006470807061745
0.2705263157894737 2
0.034537307554750166
0.21842105263157896 2
0.027322601234164464
0.1663157894736842 2
0.030144004618070857
0.11421052631578943 2
0.012699785425516495
0.06210526315789466 2
0.025748368157650966
0.01 2
0.02009811905297378
